# Bank Marketing naudojant TensorFlow **neuroninį tinklą**

Užduotis **1.3** kurso *Gilusis mokymas* (VILNIUS TECH): suprasti dėstytojo Colab užrašus ir perdaryti juos su savo arba Kaggle / viešais verslo duomenimis.

**Originalas.** Algirdas Laukaitis, *AdventureWorks DW using TensorFlow Neural Network* (`AdventureWorksDW_NN.ipynb`).  
Colab: [AdventureWorks DW using TensorFlow Neural Network](https://colab.research.google.com/drive/1LLFM2Ba5jAZSna8JnS44frwSLW9mx5As).  
Duomenys: [`vTargetMail.csv`](https://raw.githubusercontent.com/aalgirdas/AdventureWorksDW/main/vTargetMail.csv), žymė `BikeBuyer`.

**Ši versija** kartoja tuos pačius langelius ta pačia tvarka (1.1–4 ir inferencija), bet vietoj dviračių pirkėjų prognozuoja, ar banko klientas užsisakys **terminuotą indėlį**. TensorBoard ir Colab `eval_js` langeliai praleisti — jie skirti tik Colab aplinkai.


*Algirdas Laukaitis (originalas) → Bank Marketing replika*

# <font color='Green'>Bank Marketing using TensorFlow <b>Neural Network</b></font>

Originalo schema: [AdventureWorks DW schema](https://github.com/aalgirdas/AdventureWorksDW/blob/main/AdventureWorksDW2008.png?raw=true)

Šio uždavinio duomenys: [UCI Bank Marketing](https://archive.ics.uci.edu/dataset/222/bank+marketing) (Moro et al., 2014). Kampanija — telefono skambučiai Portugalijos banke; klasė `y` = ar klientas užsisakė terminuotą indėlį (`yes` / `no`).


<br><br>

---

# <font color='Green'>Step 1. From a csv data file that is available online to 3 DataFrame tables:  training, validation, and test</font>

CSV failas įkeliamas į `pandas.DataFrame`. Tada paliekami tik modeliui naudingi požymiai ir atsitiktinai padalijama į mokymo (80 %), validacijos (10 %) ir testavimo (10 %) aibes — lygiai taip, kaip originale su `vTargetMail`.


## 1.1 Įkeliame `bank-additional-full.csv` į DataFrame

Originalas skaitė GitHub CSV su `pd.read_csv(..., sep='\t')` (18 484 eilutės, 32 stulpeliai). Bank Marketing UCI faile skirtukas yra `;`, o eilučių daugiau (~41 188).

Pirmiausia skaitome vietinę kopiją `data/bank-additional-full.csv`, kad užrašai veiktų ir be interneto (pvz. Colab be Kaggle prisijungimo). Jei failo nėra, atsisiunčiame oficialų UCI `bank-additional.zip`.


In [1]:
from pathlib import Path
import io
import zipfile
import urllib.request

import pandas as pd

UCI_ZIP_URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00222/bank-additional.zip"
LOCAL_CANDIDATES = [
    Path("data/bank-additional-full.csv"),
    Path("1.3/data/bank-additional-full.csv"),
]


def load_bank_marketing():
    for path in LOCAL_CANDIDATES:
        if path.is_file():
            print(f"Skaitome vietinį failą: {path.resolve()}")
            return pd.read_csv(path, sep=";")

    print(f"Vietinio failo nėra — atsisiunčiame UCI zip: {UCI_ZIP_URL}")
    with urllib.request.urlopen(UCI_ZIP_URL, timeout=60) as resp:
        raw = io.BytesIO(resp.read())
    with zipfile.ZipFile(raw) as zf:
        with zf.open("bank-additional/bank-additional-full.csv") as handle:
            return pd.read_csv(handle, sep=";")


df_all_col = load_bank_marketing()
print("The dataframe has {} rows and {} columns.".format(df_all_col.shape[0], df_all_col.shape[1]))


Skaitome vietinį failą: /workspace/1.3/data/bank-additional-full.csv
The dataframe has 41188 rows and 21 columns.


In [2]:
df_all_col


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41183,73,retired,married,professional.course,no,yes,no,cellular,nov,fri,...,1,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,yes
41184,46,blue-collar,married,professional.course,no,no,no,cellular,nov,fri,...,1,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,no
41185,56,retired,married,university.degree,no,yes,no,cellular,nov,fri,...,2,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,no
41186,44,technician,married,professional.course,no,no,no,cellular,nov,fri,...,1,999,0,nonexistent,-1.1,94.767,-50.8,1.028,4963.6,yes


## 1.2 Peržiūrime vieną eilutę, kad suprastume, kurie požymiai tinka uždaviniui

Kaip originale, spausdiname ne pirmą, o 10-ąją eilutę. Taip pamatome ir kategorinius stulpelius (`job`, `marital`, `education`, …), ir skaitinius (`age`, `campaign`, makroekonomika), ir žymę `y`.


In [3]:
print(df_all_col.iloc[10])  # The first row of the DataFrame df.  (originale — 10-oji eilutė)


age                        41
job               blue-collar
marital               married
education             unknown
default               unknown
housing                    no
loan                       no
contact             telephone
month                     may
day_of_week               mon
duration                   55
campaign                    1
pdays                     999
previous                    0
poutcome          nonexistent
emp.var.rate              1.1
cons.price.idx         93.994
cons.conf.idx           -36.4
euribor3m               4.857
nr.employed            5191.0
y                          no
Name: 10, dtype: object


## 1.3 Pašaliname nereikalingus požymius ir pervadiname žymę į `target`

> *And we rename some attributes because we want to reuse the code snippets from official TensorFlow tutorials. Renaming an attribute is much easier than rewriting code.*

Originalas išmetė identifikatorius (vardus, raktus, adresus, telefonus) ir `BikeBuyer` pervadino į `target`. Čia analogiškai `y` tampa `target` (0 = ne, 1 = taip).

### Kodėl būtina išmesti `duration`?

`duration` — paskutinio skambučio trukmė sekundėmis. Ji paaiškėja tik **po** pokalbio, o ar klientas užsisakė indėlį (`y`) taip pat nustatoma to pokalbio pabaigoje. Ilgas pokalbis labai stipriai susijęs su `y=yes`, todėl tinklas išmoktų „nurašyti“ atsakymą iš ateities informacijos. Tai **tikslinės žymės nutekėjimas** (*target leakage*). Produkcijoje, prieš skambinant, trukmė dar nežinoma, todėl šio stulpelio naudoti negalima.

Makroekonominius vardus su taškais (`emp.var.rate`) pakeičiame pabraukimais, nes Keras `Input` varduose taškas nepatogus.


In [4]:
df = df_all_col.copy()

# duration nutekina žymę y — žr. paaiškinimą markdown langelyje aukščiau
df.drop(columns=["duration"], inplace=True)

df.rename(
    columns={
        "y": "target",
        "emp.var.rate": "emp_var_rate",
        "cons.price.idx": "cons_price_idx",
        "cons.conf.idx": "cons_conf_idx",
        "nr.employed": "nr_employed",
    },
    inplace=True,
)

df["target"] = (df["target"] == "yes").astype("int64")


In [5]:
print(df.iloc[0])
print("---------------")
print(df.dtypes)
print("---------------")
print("Klasės (0 = ne, 1 = taip):")
print(df["target"].value_counts())
print(df["target"].value_counts(normalize=True).round(4))


age                        56
job                 housemaid
marital               married
education            basic.4y
default                    no
housing                    no
loan                       no
contact             telephone
month                     may
day_of_week               mon
campaign                    1
pdays                     999
previous                    0
poutcome          nonexistent
emp_var_rate              1.1
cons_price_idx         93.994
cons_conf_idx           -36.4
euribor3m               4.857
nr_employed            5191.0
target                      0
Name: 0, dtype: object
---------------
age                 int64
job                   str
marital               str
education             str
default               str
housing               str
loan                  str
contact               str
month                 str
day_of_week           str
campaign            int64
pdays               int64
previous            int64
poutcome              s

## 1.4 Padalijame DataFrame į mokymo, validacijos ir testavimo aibes

Kai mokome mašininio mokymosi modelį, turime patikrinti, ar jis veikia su **naujai** nematytais duomenimis. Todėl duomenis dalijame į tris aibes:

- **mokymo** aibė moko svorius;
- **validacijos** aibė padeda stebėti persimokymą (*overfitting*) mokymo metu;
- **testavimo** aibė — galutinis, vieną kartą naudojamas įvertinimas.

Santykis toks pat kaip originale: **80 : 10 : 10**. Originalas kvietė `np.split(df.sample(frac=1), ...)`; dabartiniame pandas tai grąžina `ndarray`, todėl čia dalijame su `iloc`, kad liktų DataFrame. Pridedame `random_state=42`, kad padalijimas būtų atkartojamas.

### Klasės disbalansas

Teigiama klasė (`target=1`, indėlis užsakytas) sudaro tik apie **11 %** eilučių. Grynas *accuracy* čia klaidina: konstanta „visada ne“ jau duotų ~89 %. Todėl šalia tikslumo skaičiuosime **AUC**, o `model.fit` paduosime `class_weight`, kad retesnė klasė nebūtų ignoruojama.


In [6]:
import numpy as np

SEED = 42
np.random.seed(SEED)

# The dataset is in a single pandas DataFrame. Split it into training, validation, and test sets using a, for example, 80:10:10 ratio, respectively.
# Originalas: np.split(df.sample(frac=1), [int(0.8*len(df)), int(0.9*len(df))]).
# pandas 3 / NumPy 2 grąžina ndarray, todėl naudojame iloc, kad išsaugotume DataFrame.
df_shuffled = df.sample(frac=1, random_state=SEED)  # frac=1 includes all rows, randomly ordered
n = len(df_shuffled)
train = df_shuffled.iloc[: int(0.8 * n)].copy()
validation = df_shuffled.iloc[int(0.8 * n) : int(0.9 * n)].copy()
test = df_shuffled.iloc[int(0.9 * n) :].copy()
print(len(train), "training examples")
print(len(validation), "validation examples")
print(len(test), "test examples")

neg, pos = np.bincount(train["target"])
total = neg + pos
weight_for_0 = (1.0 / neg) * (total / 2.0)
weight_for_1 = (1.0 / pos) * (total / 2.0)
class_weight = {0: float(weight_for_0), 1: float(weight_for_1)}
print("class_weight:", {k: round(v, 3) for k, v in class_weight.items()})
print(
    "Mokymo aibėje teigiama klasė: "
    f"{pos}/{total} = {pos / total:.1%}"
)


32950 training examples
4119 validation examples
4119 test examples
class_weight: {0: 0.563, 1: 4.462}
Mokymo aibėje teigiama klasė: 3692/32950 = 11.2%


<br><br>

---


# <font color='Green'> Step 2. From DataFrame table to TensorFlow dataset data</font>

`tf.data.Dataset` yra efektyvus įvesties konvejeris: kiekviena eilutė virsta požymių žodynu ir žyme, tada duomenys maišomi ir pakuojami į paketus po 256 (kaip originale). `prefetch` leidžia ruošti kitą paketą, kol tinklas skaičiuoja esamą.

Originalo `df_to_dataset` iteravo `dataframe.items()` *po* `df.pop('target')`, todėl žymė vėl patektų tarp požymių. Čia naudojame jau apkarpytą `df.items()`, kaip dabartiniame TensorFlow vadovėlyje.


In [7]:
import os

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "-1")

import tensorflow as tf

tf.random.set_seed(SEED)


def df_to_dataset(dataframe, shuffle=True, batch_size=32):  # utility function that creates an input pipeline using TensorFlow’s tf.data module, then shuffles and batches the data
    df = dataframe.copy()
    labels = df.pop("target")
    df = {key: value.to_numpy()[:, tf.newaxis] for key, value in df.items()}

    ds = tf.data.Dataset.from_tensor_slices((dict(df), labels))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(dataframe), seed=SEED)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(batch_size)
    return ds


batch_size = 256
train_ds = df_to_dataset(train, batch_size=batch_size)
validation_ds = df_to_dataset(validation, shuffle=False, batch_size=batch_size)
test_ds = df_to_dataset(test, shuffle=False, batch_size=batch_size)


2026-10-21 05:49:27.807253: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1792561767.823364    3898 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1792561767.827483    3898 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1792561767.838295    3898 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1792561767.838319    3898 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1792561767.838320    3898 computation_placer.cc:177] computation placer alr

2026-10-21 05:49:29.617415: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


<br><br>

---


# <font color='Green'> Step 3. Model</font>

Keras *Functional API*: kiekvienas stulpelis turi savo `Input`, tada koduojamas ir viskas sujungiama. Taip tinklas priima vardinį žodyną (kaip `tf.data` paketai), o ne vieną didelę matricą.


In [8]:
###all_inputs = []
all_inputs = {}
encoded_features = []


## Categorical columns

Kategorinis požymis (pvz. `job=admin.`) pirma `StringLookup` / `IntegerLookup` sluoksniu virsta sveikuoju indeksu, tada `CategoryEncoding` jį paverčia multi-hot (vienam požymiui tai praktiškai one-hot) vektoriumi. `adapt` kviečiamas **tik ant mokymo aibės**, kad validacija ir testas nematytų ateities kategorijų statistikos.


In [9]:
def get_category_encoding_layer(name, dataset, dtype, max_tokens=None):
    if dtype == "string":  # Create a layer that turns strings into integer indices.
        # index = tf.keras.layers.StringLookup()
        index = tf.keras.layers.StringLookup(max_tokens=max_tokens)
    else:  # Otherwise, create a layer that turns integer values into integer indices.
        index = tf.keras.layers.IntegerLookup(max_tokens=max_tokens)

    feature_ds = dataset.map(lambda x, y: x[name])  # Prepare a `tf.data.Dataset` that only yields the feature.
    index.adapt(feature_ds)  # Learn the set of possible values and assign them a fixed integer index.
    encoder = tf.keras.layers.CategoryEncoding(num_tokens=index.vocabulary_size())  # Encode the integer indices.
    return lambda feature: encoder(index(feature))  # Apply multi-hot encoding to the indices. The lambda function captures the layer, so you can use them, or include them in the Keras Functional model later.


# Originalo analogai: MaritalStatus→marital, EnglishEducation→education,
# EnglishOccupation→job, HouseOwnerFlag→housing; plus kampanijos kontekstas.
categorical_cols = [
    "job",
    "marital",
    "education",
    "default",
    "housing",
    "loan",
    "contact",
    "month",
    "day_of_week",
    "poutcome",
]

for header in categorical_cols:
    categorical_col = tf.keras.Input(shape=(1,), name=header, dtype="string")
    encoding_layer = get_category_encoding_layer(name=header, dataset=train_ds, dtype="string")  # , max_tokens=5
    encoded_categorical_col = encoding_layer(categorical_col)
    ###all_inputs.append(categorical_col)
    all_inputs[header] = categorical_col
    encoded_features.append(encoded_categorical_col)


## Numerical columns

Skaitiniai požymiai (`age`, `campaign`, Euribor ir t. t.) praeina `Normalization`: mokymo aibėje apskaičiuojamas vidurkis ir dispersija, o į tinklą paduodama `(x - mean) / sqrt(var)`. Be to gradientai visiems požymiams būtų panašaus mastelio.


In [10]:
def get_normalization_layer(name, dataset):
    normalizer = tf.keras.layers.Normalization(axis=None)  # will shift and scale inputs into a distribution centered around 0 with standard deviation 1. It accomplishes this by precomputing the mean and variance of the data, and calling (input - mean) / sqrt(var) at runtime
    feature_ds = dataset.map(lambda x, y: x[name])  # Prepare a Dataset that only yields the feature.
    normalizer.adapt(feature_ds)  # Learn the statistics of the data.
    return normalizer


numeric_cols = [
    "age",
    "campaign",
    "pdays",
    "previous",
    "emp_var_rate",
    "cons_price_idx",
    "cons_conf_idx",
    "euribor3m",
    "nr_employed",
]

for header in numeric_cols:  # Numerical features.
    numeric_col = tf.keras.Input(shape=(1,), name=header)
    normalization_layer = get_normalization_layer(header, train_ds)
    encoded_numeric_col = normalization_layer(numeric_col)
    ###all_inputs.append(numeric_col)
    all_inputs[header] = numeric_col
    encoded_features.append(encoded_numeric_col)


In [11]:
all_features = tf.keras.layers.concatenate(encoded_features)


## Add layers of neurons and create the final model

Architektūra kaip originale: `Dense(64, relu)` → `Dropout(0.2)` → `Dense(32, relu)` → `Dropout(0.3)` → `Dense(1)` **be** sigmoidės. Nuostolis `BinaryCrossentropy(from_logits=True)` — skaitiškai stabiliau nei rankiniu būdu spausti sigmoidę. `adam` optimizatorius ir `accuracy`, kaip originale, **plius AUC**, nes klasės nesubalansuotos.


In [12]:
x = tf.keras.layers.Dense(64, activation="relu")(all_features)
x = tf.keras.layers.Dropout(0.2)(x)
x = tf.keras.layers.Dense(32, activation="relu")(x)
x = tf.keras.layers.Dropout(0.3)(x)
output = tf.keras.layers.Dense(1)(x)

model = tf.keras.Model(all_inputs, output)

model.compile(
    optimizer="adam",
    loss=tf.keras.losses.BinaryCrossentropy(from_logits=True),
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.AUC(name="auc", from_logits=True),
    ],
)


In [13]:
# model = tf.keras.Model(all_inputs, tf.keras.layers.Dense(1)(tf.keras.layers.concatenate(encoded_features)) )


In [14]:
from IPython.display import display

try:
    display(tf.keras.utils.plot_model(model, to_file="model.png", show_shapes=True))  # , rankdir="LR"  `rankdir='LR'` to make the graph horizontal.
except Exception as exc:
    print("plot_model nepavyko (pydot/graphviz):", exc)
    model.summary()


<br><br>

---


# <font color='Green'> Step 4. Train and evaluate the model</font>


<br>

## <font color='Green'>Train 🏃‍♂️</font>

`model.fit` 30 epochų (kaip originale) su validacijos aibe. `class_weight` padidina nuostolį teigiamai klasei, kad tinklas nesimokytų visada atsakyti „ne“.


In [15]:
history = model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=30,
    class_weight=class_weight,
    shuffle=False,
    verbose=2,
)


Epoch 1/30


129/129 - 2s - 13ms/step - accuracy: 0.8476 - auc: 0.7543 - loss: 0.5852 - val_accuracy: 0.8592 - val_auc: 0.7933 - val_loss: 0.5423


Epoch 2/30


129/129 - 0s - 3ms/step - accuracy: 0.8569 - auc: 0.7829 - loss: 0.5519 - val_accuracy: 0.8655 - val_auc: 0.7901 - val_loss: 0.4969


Epoch 3/30


...

129/129 - 0s - 3ms/step - accuracy: 0.8667 - auc: 0.8298 - loss: 0.5039 - val_accuracy: 0.8636 - val_auc: 0.7881 - val_loss: 0.4996


Epoch 29/30


129/129 - 0s - 3ms/step - accuracy: 0.8663 - auc: 0.8332 - loss: 0.5009 - val_accuracy: 0.8633 - val_auc: 0.7879 - val_loss: 0.4853


Epoch 30/30


129/129 - 0s - 3ms/step - accuracy: 0.8638 - auc: 0.8301 - loss: 0.5040 - val_accuracy: 0.8684 - val_auc: 0.7877 - val_loss: 0.4631


## <font color='Green'>Evaluate ⚖️</font>

Testuojame **vieną kartą** ant `test_ds`. Keras `BinaryAccuracy` slenkstį taiko logitams (0,5), todėl papildomai skaičiuojame tikimybini tikslumą po `sigmoid` ir **AUC** — tai svarbiau už gryną accuracy, kai ~89 % klientų indėlio neužsako.


In [16]:
from sklearn.metrics import accuracy_score, roc_auc_score

eval_results = model.evaluate(test_ds, return_dict=True)
print("Keras evaluate:", eval_results)
print("Accuracy", eval_results["accuracy"])
print("AUC", eval_results["auc"])

y_true = np.concatenate([y.numpy() for _, y in test_ds])
logits = model.predict(test_ds, verbose=0).ravel()
y_prob = tf.nn.sigmoid(logits).numpy()
y_hat = (y_prob >= 0.5).astype(int)

test_accuracy = float(accuracy_score(y_true, y_hat))
test_auc = float(roc_auc_score(y_true, y_prob))
print("Test accuracy (sigmoid, slenkstis 0.5):", round(test_accuracy, 4))
print("Test AUC:", round(test_auc, 4))
print("Teigiama klasė teste:", f"{int(y_true.sum())}/{len(y_true)} = {y_true.mean():.1%}")


 1/17 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8828 - auc: 0.8649 - loss: 0.4186

17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.8691 - auc: 0.7848 - loss: 0.4660


Keras evaluate: {'accuracy': 0.869143009185791, 'auc': 0.7848405838012695, 'loss': 0.46604838967323303}
Accuracy 0.869143009185791
AUC 0.7848405838012695


Test accuracy (sigmoid, slenkstis 0.5): 0.8424
Test AUC: 0.7848
Teigiama klasė teste: 484/4119 = 11.8%


<br><br>

---


# <font color='Green'> Step 5. Perform inference 🛠</font>

Išsaugome visą Keras modelį (įskaitant `StringLookup` / `Normalization` statistiką), vėl įkeliame ir prognozuojame vienam naujam klientui. Kadangi išėjimas — logitai, tikimybę gauname `tf.nn.sigmoid`.


## <font color='Green'> Save 💾 </font>


In [17]:
model.save("BankMarketing_NN_L64L32L1.keras")


## <font color='Green'> Load ⚙️</font>


In [18]:
import tensorflow as tf

reloaded_model = tf.keras.models.load_model("BankMarketing_NN_L64L32L1.keras")


## <font color='Green'> New data  🗃️</font>


In [19]:
sample = {
    "age": 35,
    "job": "admin.",
    "marital": "married",
    "education": "university.degree",
    "default": "no",
    "housing": "yes",
    "loan": "no",
    "contact": "cellular",
    "month": "aug",
    "day_of_week": "thu",
    "campaign": 1,
    "pdays": 999,
    "previous": 0,
    "poutcome": "nonexistent",
    "emp_var_rate": 1.4,
    "cons_price_idx": 93.444,
    "cons_conf_idx": -36.1,
    "euribor3m": 4.963,
    "nr_employed": 5228.1,
}


## <font color='Green'> Predict  🔮</font>


In [20]:
input_dict = {name: tf.convert_to_tensor([value]) for name, value in sample.items()}
predictions = reloaded_model.predict(input_dict)
prob = tf.nn.sigmoid(predictions[0])

print(
    "The likelihood of SUBSCRIBING TO A TERM DEPOSIT for this specific CLIENT is %.1f percent.  "
    % (100 * prob)
)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 175ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 195ms/step


The likelihood of SUBSCRIBING TO A TERM DEPOSIT for this specific CLIENT is 21.2 percent.  
